# B2B BNPL — Model Training Notebook
Trains **two systems**:
1. **Credit Predictor** — approves/rejects BNPL credit
2. **Demand Forecaster** — predicts next month order volume & revenue

Run all cells once. Saves `.pkl` files used by the Streamlit app.

In [9]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, classification_report,
    mean_absolute_error, r2_score, mean_absolute_percentage_error
)
import xgboost as xgb

print('✅ Libraries loaded')

✅ Libraries loaded


---
## 1 · Load Dataset

In [10]:
df = pd.read_csv('b2b_bnpl_dataset.csv')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Loaded: 10,000 rows × 42 columns


,industry,business_age_years,num_employees,business_type,gst_registered,region,annual_revenue_lakh,gross_margin_pct,operating_cash_flow_lakh,revenue_growth_yoy_pct,...,repeat_order_rate_pct,platform_gmv_lakh,last_order_days_ago,support_tickets_open,sector_bnpl_penetration_pct,macro_gdp_growth_pct,default_probability,bnpl_approved,credit_limit_lakh,credit_source
0,Retail (B2B),2.8,5,Pvt Ltd,1,Rural,966.68,27.04,242.06,-30.07,...,71.0,797.12,34,0,28,4.84,0.4695,0,0.00,none
1,Professional Services,2.5,23,Pvt Ltd,1,Tier-1,144.52,28.20,17.22,13.31,...,63.6,3.68,4,0,30,7.20,0.3011,1,14.61,platform
2,Construction,1.5,51,Sole Prop,1,Metro,53.29,13.84,3.48,30.63,...,63.1,138.92,0,0,10,5.43,0.3309,1,9.52,platform


---
## 2 · Feature Engineering for Demand Forecasting

In [11]:
# Encode order_value_trend: growing=1, stable=0, declining=-1
trend_map = {'growing': 1, 'stable': 0, 'declining': -1}
df['trend_enc'] = df['order_value_trend'].map(trend_map).fillna(0)

# Monthly revenue estimate
df['monthly_revenue_lakh'] = df['annual_revenue_lakh'] / 12

# Average monthly GMV per invoice
df['gmv_per_invoice'] = df['platform_gmv_lakh'] / (df['total_invoices_lifetime'].replace(0, 1))

# Targets
# Target 1: next month order count  = invoice_frequency_monthly * growth factor
df['next_month_orders'] = (
    df['invoice_frequency_monthly']
    * (1 + df['revenue_growth_yoy_pct'].clip(-50, 100) / 1200)
).round(2)

# Target 2: next month GMV = orders * avg invoice value
df['next_month_gmv'] = (
    df['next_month_orders'] * df['avg_invoice_value_lakh']
).round(2)

print('Targets created:')
print(df[['invoice_frequency_monthly','next_month_orders','avg_invoice_value_lakh','next_month_gmv']].describe().round(2))

Targets created:
       invoice_frequency_monthly  next_month_orders  avg_invoice_value_lakh  \
count                   10000.00           10000.00                10000.00   
mean                        6.14               6.18                    5.44   
std                         5.60               5.64                    8.74   
min                         0.25               0.25                    0.05   
25%                         2.62               2.63                    1.20   
50%                         4.47               4.51                    2.70   
75%                         7.64               7.70                    6.15   
max                        50.00              51.72                  200.00   

       next_month_gmv  
count        10000.00  
mean            32.62  
std             69.15  
min              0.08  
25%              4.72  
50%             12.19  
75%             32.17  
max           2338.09  


---
## 3 · Train Credit Predictor

In [12]:
CREDIT_FEATURES = [
    'business_age_years',
    'annual_revenue_lakh',
    'num_employees',
    'gross_margin_pct',
    'debt_to_equity_ratio',
    'current_ratio',
    'gst_registered',
    'pct_invoices_paid_on_time',
    'avg_payment_delay_days',
    'months_on_platform',
    'credit_bureau_score',
    'prev_bnpl_defaults',
    'num_credit_enquiries_6m',
    'existing_loan_lakh',
]

Xc  = df[CREDIT_FEATURES]
yc  = df['bnpl_approved']
yr  = df['default_probability']

Xc_tr, Xc_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(
    Xc, yc, yr, test_size=0.2, random_state=42, stratify=yc)

# Approval classifier
clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='auc', random_state=42, verbosity=0)
clf.fit(Xc_tr, yc_tr)
auc = roc_auc_score(yc_te, clf.predict_proba(Xc_te)[:, 1])
print(f'Credit Classifier  AUC-ROC : {auc:.4f}')
print(classification_report(yc_te, clf.predict(Xc_te), target_names=['Rejected','Approved']))

# Default risk regressor
risk_reg = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
risk_reg.fit(Xc_tr, yr_tr)
print(f'Default Risk   MAE : {mean_absolute_error(yr_te, risk_reg.predict(Xc_te)):.4f}')

Credit Classifier  AUC-ROC : 0.9827
              precision    recall  f1-score   support

    Rejected       0.93      0.89      0.91       790
    Approved       0.93      0.95      0.94      1210

    accuracy                           0.93      2000
   macro avg       0.93      0.92      0.93      2000
weighted avg       0.93      0.93      0.93      2000

Default Risk   MAE : 0.0374


---
## 4 · Train Demand Forecasting Models

In [13]:
DEMAND_FEATURES = [
    'months_on_platform',          # Platform maturity
    'invoice_frequency_monthly',   # Current order cadence
    'avg_invoice_value_lakh',      # Average order size
    'repeat_order_rate_pct',       # Loyalty signal
    'revenue_growth_yoy_pct',      # Growth momentum
    'revenue_volatility_6m_pct',   # Demand stability
    'platform_gmv_lakh',           # Historical GMV
    'num_unique_billers',          # Supplier diversity
    'last_order_days_ago',         # Recency
    'gross_margin_pct',            # Margin health
    'annual_revenue_lakh',         # Business scale
    'trend_enc',                   # Order trend direction
    'gmv_per_invoice',             # Value per transaction
    'sector_bnpl_penetration_pct', # Market context
    'macro_gdp_growth_pct',        # Macro environment
]

Xd = df[DEMAND_FEATURES]
y_orders = df['next_month_orders']
y_gmv    = df['next_month_gmv']

Xd_tr, Xd_te, yo_tr, yo_te, yg_tr, yg_te = train_test_split(
    Xd, y_orders, y_gmv, test_size=0.2, random_state=42)

# Model A: predict order count
order_reg = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
order_reg.fit(Xd_tr, yo_tr)
yo_pred = order_reg.predict(Xd_te)
print(f'Order Count  MAE  : {mean_absolute_error(yo_te, yo_pred):.4f}')
print(f'Order Count  R²   : {r2_score(yo_te, yo_pred):.4f}')
print(f'Order Count  MAPE : {mean_absolute_percentage_error(yo_te.clip(0.01), yo_pred.clip(0.01))*100:.2f}%')

# Model B: predict monthly GMV
gmv_reg = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
gmv_reg.fit(Xd_tr, yg_tr)
yg_pred = gmv_reg.predict(Xd_te)
print(f'\nGMV Forecast MAE  : {mean_absolute_error(yg_te, yg_pred):.4f}')
print(f'GMV Forecast R²   : {r2_score(yg_te, yg_pred):.4f}')

Order Count  MAE  : 0.1068
Order Count  R²   : 0.9915
Order Count  MAPE : 1.90%

GMV Forecast MAE  : 3.0275
GMV Forecast R²   : 0.8231


---
## 5 · Save All Models

In [14]:
joblib.dump(clf,              'bnpl_classifier.pkl')
joblib.dump(risk_reg,         'bnpl_risk_regressor.pkl')
joblib.dump(CREDIT_FEATURES,  'bnpl_credit_features.pkl')

joblib.dump(order_reg,        'bnpl_order_forecast.pkl')
joblib.dump(gmv_reg,          'bnpl_gmv_forecast.pkl')
joblib.dump(DEMAND_FEATURES,  'bnpl_demand_features.pkl')

print('All model files saved:')
print('  bnpl_classifier.pkl        — credit approval')
print('  bnpl_risk_regressor.pkl    — default risk score')
print('  bnpl_credit_features.pkl   — credit feature list')
print('  bnpl_order_forecast.pkl    — order count forecaster')
print('  bnpl_gmv_forecast.pkl      — GMV forecaster')
print('  bnpl_demand_features.pkl   — demand feature list')
print()
print('Now run:  streamlit run b2b_bnpl_app.py')

All model files saved:
  bnpl_classifier.pkl        — credit approval
  bnpl_risk_regressor.pkl    — default risk score
  bnpl_credit_features.pkl   — credit feature list
  bnpl_order_forecast.pkl    — order count forecaster
  bnpl_gmv_forecast.pkl      — GMV forecaster
  bnpl_demand_features.pkl   — demand feature list

Now run:  streamlit run b2b_bnpl_app.py


---
## 6 · Quick Sanity Check

In [15]:
sample_credit = Xc_te.iloc[:1]
sample_demand = Xd_te.iloc[:1]

print('=== Credit Predictor ===')
print(f'  Approved     : {joblib.load("bnpl_classifier.pkl").predict(sample_credit)[0]}')
print(f'  Approval Prob: {joblib.load("bnpl_classifier.pkl").predict_proba(sample_credit)[0,1]:.2%}')
print(f'  Default Risk : {joblib.load("bnpl_risk_regressor.pkl").predict(sample_credit)[0]:.2%}')

print('\n=== Demand Forecaster ===')
print(f'  Next Month Orders : {joblib.load("bnpl_order_forecast.pkl").predict(sample_demand)[0]:.2f}')
print(f'  Next Month GMV    : ₹{joblib.load("bnpl_gmv_forecast.pkl").predict(sample_demand)[0]:.2f}L')

print('\n✅ All models working correctly!')

=== Credit Predictor ===
  Approved     : 1
  Approval Prob: 88.88%
  Default Risk : 32.63%

=== Demand Forecaster ===
  Next Month Orders : 1.79
  Next Month GMV    : ₹2.77L

✅ All models working correctly!
